<a href="https://colab.research.google.com/github/Khuld13/ML-intern-starter/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khuld13/ML-intern-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

# I picked Lane 2: Refresh / Content Opportunity Scoring
It's about deciding which pages a content team should look at first when they only have time to review a handful — refresh, expand, protect, or leave alone.
I liked this lane because it's not about building the fanciest model, it's about being useful and honest: start with a simple, explainable baseline, then see if a real model can beat it. For now I'm treating this as a current-state triage problem (similar to the starter pipeline's is_declining_label), and I'll try to move to a future-looking version (predict decline before it happens) once the basics work.

In [16]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/work/notebooks/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(f"Total rows: {len(df)}")
print(df.columns.tolist())

Total rows: 30000
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Lane 2 candidate pages (impressions_90d > 0): {(df['impressions_90d'] > 0).sum()}")

Lane 2 candidate pages (impressions_90d > 0): 30000


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** Which pages get reviewed first this cycle, and what a reviewer should consider doing with them (refresh, expand, protect, prune, monitor).

**Actor:** A content/SEO reviewer who can only realistically get through a limited list not the whole site.

**Cost of being wrong:** If I flag a fine page as "review this," the reviewer just loses a bit of time. If I miss a page that's actually declining, nobody catches it — it keeps losing traffic quietly. Missing a real problem feels worse than a wasted review, so I'll lean toward not missing declining pages, even if that means occasionally flagging one that turns out fine. I'll revisit this once I have real numbers to look at.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
declining = (df['trend_direction'] == 'down').sum()
print(f"Pages currently trending down: {declining} out of {len(df)} total")

Pages currently trending down: 16262 out of 30000 total


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [20]:
# 1. Volume check: does the lane still have enough data after the starter pipeline's own filter?

filtered = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)]
print(f"\nRows after starter filter (impressions_90d > 0, content_age_days >= 90): {len(filtered)}")



Rows after starter filter (impressions_90d > 0, content_age_days >= 90): 30000


In [21]:
# 2. How many pages are already flagged as declining — this is your future label's ancestor

print(f"\ntrend_direction breakdown:")
print(df['trend_direction'].value_counts())


trend_direction breakdown:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [22]:
# 3. Overlap: of the declining pages, how many also have real demand (impressions)?

declining_with_demand = df[(df['trend_direction'] == 'down') & (df['impressions_90d'] >= 100)]
print(f"\nDeclining pages with impressions_90d >= 100: {len(declining_with_demand)}")


Declining pages with impressions_90d >= 100: 13152


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

This project can say "these pages show signs associated with decline or opportunity, based on past data" that's it. It can rank pages and explain why they're flagged. It can't say a refresh will fix a page, since that would need an actual experiment, not just historical data. And it can't say anything about how Google's algorithm actually works. It's a tool to help a human decide where to look first not a tool that decides for them

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
sensitive_cols = [c for c in df.columns if any(k in c.lower() for k in ['url', 'client_name', 'query', 'title'])]
print(f"Columns that look sensitive: {sensitive_cols}")

Columns that look sensitive: []


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.